# Section 2.1 — The "It Works on My Machine" Problem

This notebook makes **environmental drift** visible with a small pandas example.

The code stays identical. The environment changes. The outcome changes.

## Step 1: Run this cell in your current kernel

This code uses the monthly frequency alias `"M"`.

In pre-3.0 pandas, that alias still works. In recent pandas it has been removed in favor of `"ME"`.

In [1]:
import pandas as pd

print(f"pandas version: {pd.__version__}")

dates = pd.date_range("2020-01-01", periods=3, freq="M")
print(dates)


pandas version: 2.3.3
DatetimeIndex(['2020-01-31', '2020-02-29', '2020-03-31'], dtype='datetime64[ns]', freq='ME')


/var/folders/tl/wv0f4f7x35v2qmpcj9_8nv280000gn/T/ipykernel_99862/3884923889.py:5: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range("2020-01-01", periods=3, freq="M")


## Step 2: Switch kernels and run the same cell again

If you have a **pandas 2.3.3** kernel, switch to it and run the same code.

Expected comparison:

| | pandas 2.3.3 | pandas 3.0 |
|---|---|---|
| `freq="M"` | works, but warns that `M` is deprecated | fails and tells you to use `ME` |

If your second kernel is older than 2.3.3, the code may still run without the warning. The important point is the progression: code that once worked becomes deprecated, then eventually stops working entirely.

That is environmental drift: the notebook didn't change, but the environment did.

## Why this matters

This is exactly how notebook reproducibility problems creep in:

- you write code when a library API still works
- a later version marks it as deprecated
- a later version removes it altogether
- the same notebook now warns or breaks on someone else's machine

Nothing about the notebook file changed. Only the environment changed.

## Optional: compare from inside this notebook

If you have another Python interpreter on disk, the cell below will try to run the same snippet there and print the result.

In [ ]:
from pathlib import Path
import subprocess
import sys
import textwrap

candidate_pythons = [
    Path('Module_2/env_py310_pandas153/bin/python'),
    Path('env_py310_pandas153/bin/python'),
]

script = textwrap.dedent('''
import pandas as pd
print(f"pandas version: {pd.__version__}")
dates = pd.date_range("2020-01-01", periods=3, freq="M")
print(dates)
''')

print('Current notebook environment:')
print(f'python: {sys.executable}')
print()

for candidate in candidate_pythons:
    if not candidate.exists():
        continue

    print(f'Trying second environment: {candidate}')
    result = subprocess.run(
        [str(candidate), '-c', script],
        capture_output=True,
        text=True,
    )

    if result.stdout:
        print('stdout:')
        print(result.stdout)

    if result.stderr:
        print('stderr:')
        print(result.stderr)

    if result.returncode == 0:
        break
else:
    print('No prepared second environment found locally.')
